# Brain Tumor MRI Classifier - Training Notebook

Transfer learning with EfficientNet-B0 on the [Brain Tumor MRI Dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset).

**Classes:** glioma, meningioma, no_tumor, pituitary

Run this notebook in Google Colab with a GPU runtime.

## 1. Setup & Download Dataset

In [ ]:
# Install kaggle CLI if needed
!pip install -q kaggle

In [ ]:
import os

# Paste your Kaggle API token here (from kaggle.com/settings > API)
os.environ["KAGGLE_API_TOKEN"] = "YOUR_KAGGLE_TOKEN_HERE"

In [ ]:
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset
!unzip -q brain-tumor-mri-dataset.zip -d data

In [ ]:
import os

# Check dataset structure
for split in ['Training', 'Testing']:
    print(f"\n{split}:")
    split_dir = os.path.join('data', split)
    for cls in sorted(os.listdir(split_dir)):
        cls_dir = os.path.join(split_dir, cls)
        if os.path.isdir(cls_dir):
            print(f"  {cls}: {len(os.listdir(cls_dir))} images")

## 2. Data Loading & Augmentation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# ImageNet normalization stats
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = datasets.ImageFolder('data/Training', transform=train_transforms)
test_dataset = datasets.ImageFolder('data/Testing', transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

CLASS_NAMES = train_dataset.classes
print(f"Classes: {CLASS_NAMES}")
print(f"Train: {len(train_dataset)} images")
print(f"Test: {len(test_dataset)} images")

In [ ]:
# Visualize some samples
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i * (len(train_dataset) // 8)]
    # Denormalize for display
    img_display = img.permute(1, 2, 0).numpy()
    img_display = img_display * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    img_display = np.clip(img_display, 0, 1)
    ax.imshow(img_display)
    ax.set_title(CLASS_NAMES[label])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Model Setup

In [ ]:
# Load pretrained EfficientNet-B0
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Freeze backbone
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier head
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(1280, 4),
)

model = model.to(device)

# Count parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")

## 4. Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

NUM_EPOCHS = 15
train_losses = []
test_accuracies = []

for epoch in range(NUM_EPOCHS):
    # Train
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss = running_loss / total
    train_acc = 100.0 * correct / total
    train_losses.append(train_loss)

    # Evaluate
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    test_acc = 100.0 * correct / total
    test_accuracies.append(test_acc)

    scheduler.step(train_loss)
    lr = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {train_loss:.4f} | Train Acc: {train_acc:.1f}% | Test Acc: {test_acc:.1f}% | LR: {lr:.6f}")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses)
ax1.set_title('Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')

ax2.plot(test_accuracies)
ax2.set_title('Test Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')

plt.tight_layout()
plt.show()

print(f"\nBest test accuracy: {max(test_accuracies):.1f}%")

## 5. Detailed Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

## 6. Export Model

In [ ]:
# Save model state dict (CPU)
model_cpu = model.cpu()
torch.save(model_cpu.state_dict(), 'tumor_classifier.pth')

# Check file size
size_mb = os.path.getsize('tumor_classifier.pth') / (1024 * 1024)
print(f"Model saved: tumor_classifier.pth ({size_mb:.1f} MB)")

In [ ]:
# Download the model file
files.download('tumor_classifier.pth')

In [ ]:
# Also save a few sample images for the frontend demo
import shutil
os.makedirs('sample_mris', exist_ok=True)

for cls in CLASS_NAMES:
    cls_dir = os.path.join('data', 'Testing', cls)
    sample = os.listdir(cls_dir)[0]
    src = os.path.join(cls_dir, sample)
    dst = os.path.join('sample_mris', f'{cls}.jpg')
    shutil.copy2(src, dst)
    print(f"Saved sample: {dst}")

# Zip and download samples
shutil.make_archive('sample_mris', 'zip', '.', 'sample_mris')
files.download('sample_mris.zip')